Carga en un notebook nuevo su archivo limpio de la Play Store y resuelvan los siguientes tres retos de ingeniería. Documenten su código y respondan a las preguntas de negocio en celdas Markdown.

In [46]:
import pandas as pd

df = pd.read_csv('../unit-II/Playstore_cleaned.csv')
df.describe()

,Rating,Reviews,Installs,Price
count,8180.000000,9.636000e+03,9.636000e+03,9636.000000
mean,4.173753,2.171086e+05,7.796048e+06,0.336586
std,0.536736,1.833474e+06,5.382106e+07,1.899534
min,1.000000,0.000000e+00,0.000000e+00,0.000000
25%,4.000000,2.500000e+01,1.000000e+03,0.000000
50%,4.300000,9.790000e+02,1.000000e+05,0.000000
75%,4.500000,2.949750e+04,1.000000e+06,0.000000
max,5.000000,7.815831e+07,1.000000e+09,46.990000


**Reto 1**: La Trampa de las Unidades de Medida

El equipo de infraestructura quiere saber si las aplicaciones más pesadas tienen menos descargas. Para averiguarlo, necesitan que la columna Size sea completamente numérica. Sin embargo, si revisan la columna, encontrarán valores como "19M" (Megabytes), "201k" (Kilobytes) y un texto molesto que dice "Varies with device".

**Tu misión algorítmica**:

a. Escribe una función pura en Python que reciba un string. Si el string termina en 'M', quítale la 'M' y conviértelo a flotante (dejándolo como Megabytes). Si termina en 'k', quítale la 'k', conviértelo a flotante y divídelo entre 1024 (para pasarlo también a Megabytes). Si dice "Varies with device", devuélvelo como un valor nulo de Numpy (np.nan).

b. Aplica esta función a toda la columna utilizando el método .apply().

**Pregunta a responder**: Una vez convertida la columna a valores numéricos (Megabytes), ejecuta el método .mean(). ¿Cuál es el peso promedio en Megabytes de las apps en la Play Store?

In [47]:
import numpy as np
def clean_size(size):
    if 'M' in size:
        return float(size.replace('M', ''))
    if 'K' in size:
        return float(size.replace('K', ''))/1024
    if size == 'Varies with device':
        return np.nan

df['Size'] = df['Size'].apply(clean_size)

df['Size'].mean()


np.float64(21.19225543478261)

**R** = 21.19 M

**Reto 2**: El Tipo de Dato Cronológico

Ningún análisis de software está completo sin analizar el tiempo. La columna Last Updated tiene fechas escritas como texto: "January 7, 2018". Para un modelo matemático o una serie de tiempo, eso es texto inservible.

**Tu misión algorítmica:**

a. Investiga y utiliza la función pd.to_datetime() de Pandas para sobrescribir la columna Last Updated, convirtiéndola del tipo string (Object) al tipo nativo datetime64.

b. Ahora que es un objeto de tiempo, Pandas te permite extraer componentes específicos. Crea una nueva columna llamada Year_Updated extrayendo únicamente el año (df['Last Updated'].dt.year).

**Pregunta a responder**: Utilizando la sumarización categórica (value_counts()) sobre tu nueva columna Year_Updated, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

In [48]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'])
# df['Last Updated']
df['Year_Updated'] = df['Last Updated'].dt.year
# df

df.value_counts('Year_Updated')

Year_Updated
2018    6271
2017    1785
2016     779
2015     448
2014     203
2013     108
2012      26
2011      15
2010       1
Name: count, dtype: int64

**R** = 2018 con 6271 actualizaciones registradas

**Reto 3**: La Decisión Arquitectónica

Al resolver el Reto 1, introdujiste intencionalmente valores NaN en las aplicaciones cuyo tamaño decía "Varies with device".

**Tu misión algorítmica**: Evalúa cuántos registros quedaron vacíos. Como ingenieros, decidan y apliquen la mejor técnica: ¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?

**Pregunta a responder**: Redacten una breve justificación técnica de 3 líneas explicando qué método eligieron y por qué lo consideran superior para no dañar al modelo de predicción.

In [49]:
df['Size'].isna().sum()


np.int64(1540)

In [50]:
sizes_median_by_cat = (df['Size'].groupby(df['Category']).median()).to_dict()
# print(sizes_median_by_cat)

def fill_size(size, category):
    if np.isnan(size):
        return sizes_median_by_cat[category]
    return size

df['Size'] = df.apply(lambda x: fill_size(x['Size'], x['Category']), axis=1)

df['Size'].isna().sum()


np.int64(0)

**R** = me parece más apropiado hacer una imputación de valores, esto más que nada por la cantidad de registros que se estarían borrando del dataset. Si se hace el borrado se estaría eliminando casi el 20% de los registros con los que se cuenta para la creación del modelo, lo que podía terminar introduciendo sesgos de forma indirecta. Tal vez no usuaría la mediana global, sino la mediana por categoría para tratar de introducir valores un poco más "acertados", pero en general preferiría la imputación sobre el borrado.